# Quandela Swaptions Challenge - Working Notebook

This notebook is adapted to this repo layout and auto-detects the dataset folder inside `quandela_folder`.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

np.random.seed(0)
pd.set_option('display.max_columns', 20)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == 'notebooks' else cwd

candidate_dirs = [
    PROJECT_ROOT / 'Quandela_folder' / 'CHALLENGE RESOURCES' / 'DATASETS',
    PROJECT_ROOT / 'quandela_folder' / 'CHALLENGE RESOURCES' / 'DATASETS',
    PROJECT_ROOT,
]

def find_file(name: str) -> Path:
    for d in candidate_dirs:
        p = d / name
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {name}. Checked: {[str(d) for d in candidate_dirs]}")

TRAIN_XLSX = find_file('train.xlsx')
TEMPLATE_XLSX = find_file('test_template.xlsx')
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRAIN_XLSX:', TRAIN_XLSX)
print('TEMPLATE_XLSX:', TEMPLATE_XLSX)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
train_df = pd.read_excel(TRAIN_XLSX)
train_df['Date'] = pd.to_datetime(train_df['Date'], dayfirst=True)
train_df = train_df.sort_values('Date').reset_index(drop=True)

surface_cols = [c for c in train_df.columns if c != 'Date']
assert len(surface_cols) == 224, f'Expected 224 surface columns, got {len(surface_cols)}'
assert train_df[surface_cols].isna().sum().sum() == 0, 'Train contains NaNs.'

print('train shape:', train_df.shape)
print('date range:', train_df['Date'].min().date(), '->', train_df['Date'].max().date())
print('surface columns:', len(surface_cols))
train_df.head(2)


In [ ]:
def parse_tenor_maturity(col: str):
    # Supports labels like 'Tenor: 1; Maturity: 0.5' and fallback numeric extraction.
    parts = [p.strip() for p in col.split(';')]
    if len(parts) >= 2 and ':' in parts[0] and ':' in parts[1]:
        t = float(parts[0].split(':', 1)[1].strip())
        m = float(parts[1].split(':', 1)[1].strip())
        return t, m
    nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", col)
    if len(nums) < 2:
        raise ValueError(f'Cannot parse tenor/maturity from column: {col}')
    return float(nums[0]), float(nums[1])

pairs = [parse_tenor_maturity(c) for c in surface_cols]
tenors = sorted({t for t, _ in pairs})
maturities = sorted({m for _, m in pairs})
assert len(tenors) * len(maturities) == 224

print('grid:', len(tenors), 'x', len(maturities))
print('tenor sample:', tenors[:5])
print('maturity sample:', maturities[:5])


In [ ]:
Y = train_df[surface_cols].values.astype(float)

# Full-data preprocessing for final forecasting/submission model.
scaler = StandardScaler()
Yz = scaler.fit_transform(Y)

k = min(12, Y.shape[1], max(2, Y.shape[0] // 20))
pca = PCA(n_components=k, random_state=0)
F = pca.fit_transform(Yz)
print('PCA k =', k, '| explained variance =', float(pca.explained_variance_ratio_.sum()))

def make_window_dataset(F_values, W: int, H: int):
    T, _ = F_values.shape
    X_list, y_list = [], []
    for t in range(W - 1, T - H):
        X_list.append(F_values[t - W + 1:t + 1].reshape(-1))
        y_list.append(F_values[t + H])
    if not X_list:
        raise ValueError('Not enough rows for selected window/horizon.')
    return np.vstack(X_list), np.vstack(y_list)

W = min(20, max(5, len(F) // 8))
H = 1
X1, y1 = make_window_dataset(F, W=W, H=H)
split_idx = int(0.8 * len(X1))
train_target_last_idx = W + split_idx - 1
print('window dataset:', X1.shape, y1.shape, '| W =', W, '| split_idx =', split_idx)

# Leakage-free preprocessing for validation diagnostics.
scaler_eval = StandardScaler()
scaler_eval.fit(Y[:train_target_last_idx + 1])
Yz_eval = scaler_eval.transform(Y)

pca_eval = PCA(n_components=k, random_state=0)
pca_eval.fit(Yz_eval[:train_target_last_idx + 1])
F_eval = pca_eval.transform(Yz_eval)
X1_eval, y1_eval = make_window_dataset(F_eval, W=W, H=H)

X_tr_eval, X_va_eval = X1_eval[:split_idx], X1_eval[split_idx:]
y_tr_eval, y_va_eval = y1_eval[:split_idx], y1_eval[split_idx:]

def cv_score_ridge(X_values, y_values, alphas=(1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100, 1e3), n_splits=5):
    n_splits = min(n_splits, max(2, len(X_values) - 1))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    best = None
    for a in alphas:
        mses = []
        for tr, va in tscv.split(X_values):
            model = Ridge(alpha=a)
            model.fit(X_values[tr], y_values[tr])
            pred = model.predict(X_values[va])
            mses.append(mean_squared_error(y_values[va], pred))
        mse = float(np.mean(mses))
        if best is None or mse < best[0]:
            best = (mse, a)
    return best

def predict_factors_iterative(F_hist, steps: int, W: int, ridge_model):
    F_ext = F_hist.copy()
    for _ in range(steps):
        x = F_ext[-W:].reshape(1, -1)
        f_next = ridge_model.predict(x)
        F_ext = np.vstack([F_ext, f_next])
    return F_ext[-steps:]

def predict_factors_iterative_naive(F_hist, steps: int):
    F_ext = F_hist.copy()
    for _ in range(steps):
        f_next = F_ext[-1:].copy()
        F_ext = np.vstack([F_ext, f_next])
    return F_ext[-steps:]

def factors_to_surface(F_fut):
    Yz_fut = pca.inverse_transform(F_fut)
    return scaler.inverse_transform(Yz_fut)

def factors_to_surface_eval(F_fut):
    Yz_fut = pca_eval.inverse_transform(F_fut)
    return scaler_eval.inverse_transform(Yz_fut)

best_mse, best_alpha = cv_score_ridge(X_tr_eval, y_tr_eval)
classical_eval_model = Ridge(alpha=best_alpha)
classical_eval_model.fit(X_tr_eval, y_tr_eval)

pred_va_classical_eval = classical_eval_model.predict(X_va_eval)
naive_va_eval = X_va_eval[:, -k:]
mean_va_eval = np.repeat(y_tr_eval.mean(axis=0, keepdims=True), len(y_va_eval), axis=0)

y_va_surface_eval = factors_to_surface_eval(y_va_eval)
pred_va_surface_eval = factors_to_surface_eval(pred_va_classical_eval)
naive_va_surface_eval = factors_to_surface_eval(naive_va_eval)
mean_va_surface_eval = factors_to_surface_eval(mean_va_eval)

metrics_df = pd.DataFrame({
    'Model': ['Classical Ridge', 'Naive persistence', 'Mean predictor'],
    'Factor_MSE': [
        mean_squared_error(y_va_eval, pred_va_classical_eval),
        mean_squared_error(y_va_eval, naive_va_eval),
        mean_squared_error(y_va_eval, mean_va_eval),
    ],
    'Factor_MAE': [
        mean_absolute_error(y_va_eval, pred_va_classical_eval),
        mean_absolute_error(y_va_eval, naive_va_eval),
        mean_absolute_error(y_va_eval, mean_va_eval),
    ],
    'Surface_MSE': [
        mean_squared_error(y_va_surface_eval, pred_va_surface_eval),
        mean_squared_error(y_va_surface_eval, naive_va_surface_eval),
        mean_squared_error(y_va_surface_eval, mean_va_surface_eval),
    ],
    'Surface_MAE': [
        mean_absolute_error(y_va_surface_eval, pred_va_surface_eval),
        mean_absolute_error(y_va_surface_eval, naive_va_surface_eval),
        mean_absolute_error(y_va_surface_eval, mean_va_surface_eval),
    ],
})

try:
    display(metrics_df.round(8))
except NameError:
    print(metrics_df.round(8).to_string(index=False))

F_hist_eval_train = F_eval[:train_target_last_idx + 1]
F_true_recursive_eval = F_eval[train_target_last_idx + 1:train_target_last_idx + 1 + len(y_va_eval)]
F_pred_recursive_eval = predict_factors_iterative(F_hist_eval_train, steps=len(y_va_eval), W=W, ridge_model=classical_eval_model)
F_pred_recursive_naive = predict_factors_iterative_naive(F_hist_eval_train, steps=len(y_va_eval))

recursive_df = pd.DataFrame({
    'Model': ['Classical Ridge (recursive)', 'Naive persistence (recursive)'],
    'Factor_MSE': [
        mean_squared_error(F_true_recursive_eval, F_pred_recursive_eval),
        mean_squared_error(F_true_recursive_eval, F_pred_recursive_naive),
    ],
    'Factor_MAE': [
        mean_absolute_error(F_true_recursive_eval, F_pred_recursive_eval),
        mean_absolute_error(F_true_recursive_eval, F_pred_recursive_naive),
    ],
    'Surface_MSE': [
        mean_squared_error(factors_to_surface_eval(F_true_recursive_eval), factors_to_surface_eval(F_pred_recursive_eval)),
        mean_squared_error(factors_to_surface_eval(F_true_recursive_eval), factors_to_surface_eval(F_pred_recursive_naive)),
    ],
    'Surface_MAE': [
        mean_absolute_error(factors_to_surface_eval(F_true_recursive_eval), factors_to_surface_eval(F_pred_recursive_eval)),
        mean_absolute_error(factors_to_surface_eval(F_true_recursive_eval), factors_to_surface_eval(F_pred_recursive_naive)),
    ],
})

print('Validation diagnostics (no leakage):')
try:
    display(recursive_df.round(8))
except NameError:
    print(recursive_df.round(8).to_string(index=False))

# Final model for submission: retrain on all available windows using alpha selected on clean validation setup.
ridge = Ridge(alpha=best_alpha)
ridge.fit(X1, y1)
print('best alpha (selected on clean split):', best_alpha, '| cv mse:', best_mse)
print('final classical model fitted on full windows:', X1.shape, y1.shape)


In [ ]:
template_df = pd.read_excel(TEMPLATE_XLSX)
tmpl_surface_cols = [c for c in template_df.columns if c not in ('Type', 'Date')]

missing_cols = set(surface_cols) - set(tmpl_surface_cols)
if missing_cols:
    raise ValueError(f'Template is missing {len(missing_cols)} columns. Example: {list(missing_cols)[:3]}')

n_rows = len(template_df)
F_future = predict_factors_iterative(F, steps=n_rows, W=W, ridge_model=ridge)
Y_future = factors_to_surface(F_future)

pred_df = pd.DataFrame(Y_future, columns=surface_cols)[tmpl_surface_cols]
submission_df = template_df.copy()
submission_df.loc[:, tmpl_surface_cols] = pred_df.values

submission_path = OUTPUT_DIR / 'submission_baseline_notebook.xlsx'
submission_df.to_excel(submission_path, index=False)
print('submission saved to:', submission_path)
submission_df.head(2)


## Visualization
Quick plots to inspect the training surfaces and baseline forecast output.


In [ ]:
import matplotlib.pyplot as plt

def column_for(tenor_target: float, maturity_target: float):
    best_col = None
    best_dist = float("inf")
    for c in surface_cols:
        t, m = parse_tenor_maturity(c)
        d = abs(t - tenor_target) + abs(m - maturity_target)
        if d < best_dist:
            best_dist = d
            best_col = c
    return best_col

series_specs = [(1, 1), (5, 5), (10, 10), (5, 1)]
fig, axes = plt.subplots(len(series_specs), 1, figsize=(12, 10), sharex=True)
for ax, (tt, mm) in zip(axes, series_specs):
    col = column_for(tt, mm)
    ax.plot(train_df["Date"], train_df[col], lw=1.5)
    ax.set_title(f"{col}")
    ax.set_ylabel("Price")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

pair_map = {c: parse_tenor_maturity(c) for c in surface_cols}
tenor_idx = {t: i for i, t in enumerate(sorted({tm[0] for tm in pair_map.values()}))}
maturity_idx = {m: j for j, m in enumerate(sorted({tm[1] for tm in pair_map.values()}))}
col_to_idx = {c: i for i, c in enumerate(surface_cols)}

def row_to_surface(row_values):
    surf = np.full((len(tenor_idx), len(maturity_idx)), np.nan)
    for c in surface_cols:
        t, m = pair_map[c]
        surf[tenor_idx[t], maturity_idx[m]] = row_values[c]
    return surf

def array_to_surface(values):
    surf = np.full((len(tenor_idx), len(maturity_idx)), np.nan)
    values = np.asarray(values)
    for c in surface_cols:
        t, m = pair_map[c]
        surf[tenor_idx[t], maturity_idx[m]] = values[col_to_idx[c]]
    return surf

def _set_surface_axes(axs, shape):
    for ax in axs:
        ax.set_xlabel("Maturity index")
        ax.set_ylabel("Tenor index")
        ax.set_xticks(np.arange(shape[1]))
        ax.set_yticks(np.arange(shape[0]))

def plot_reference_vs_prediction(
    ref_surface,
    pred_surface,
    ref_title="Reference Surface",
    pred_title="Predicted Surface",
    diff_title_prefix="Absolute Difference",
):
    diff_surface = pred_surface - ref_surface
    abs_diff = np.abs(diff_surface)

    vmin = min(np.nanmin(ref_surface), np.nanmin(pred_surface))
    vmax = max(np.nanmax(ref_surface), np.nanmax(pred_surface))
    diff_max = np.nanmax(abs_diff)
    mae = np.nanmean(abs_diff)
    rmse = np.sqrt(np.nanmean(diff_surface ** 2))

    fig, axs = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
    axs[0].imshow(ref_surface, aspect="auto", interpolation="nearest", cmap="viridis", vmin=vmin, vmax=vmax)
    axs[0].set_title(ref_title)

    im1 = axs[1].imshow(pred_surface, aspect="auto", interpolation="nearest", cmap="viridis", vmin=vmin, vmax=vmax)
    axs[1].set_title(pred_title)

    im2 = axs[2].imshow(abs_diff, aspect="auto", interpolation="nearest", cmap="magma", vmin=0, vmax=diff_max)
    axs[2].set_title(f"{diff_title_prefix} | MAE={mae:.4f}, RMSE={rmse:.4f}")

    _set_surface_axes(axs, ref_surface.shape)
    cb_price = fig.colorbar(im1, ax=axs[:2], shrink=0.9)
    cb_price.set_label("Price")
    cb_err = fig.colorbar(im2, ax=axs[2], shrink=0.9)
    cb_err.set_label("|Pred - Ref|")
    plt.show()

def plot_classical_vs_quantum(c_surface, q_surface):
    diff_surface = q_surface - c_surface
    vmin = min(np.nanmin(c_surface), np.nanmin(q_surface))
    vmax = max(np.nanmax(c_surface), np.nanmax(q_surface))
    dmax = np.nanmax(np.abs(diff_surface))
    dmean = np.nanmean(diff_surface)
    dmae = np.nanmean(np.abs(diff_surface))

    fig, axs = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    axs[0].imshow(c_surface, aspect="auto", interpolation="nearest", cmap="viridis", vmin=vmin, vmax=vmax)
    axs[0].set_title("Classical First Forecast")

    im1 = axs[1].imshow(q_surface, aspect="auto", interpolation="nearest", cmap="viridis", vmin=vmin, vmax=vmax)
    axs[1].set_title("Quantum First Forecast")

    im2 = axs[2].imshow(diff_surface, aspect="auto", interpolation="nearest", cmap="RdBu_r", vmin=-dmax, vmax=dmax)
    axs[2].set_title(f"Quantum - Classical | mean={dmean:.4f}, MAE={dmae:.4f}")

    _set_surface_axes(axs, c_surface.shape)
    cb_price = fig.colorbar(im1, ax=axs[:2], shrink=0.9)
    cb_price.set_label("Price")
    cb_delta = fig.colorbar(im2, ax=axs[2], shrink=0.9)
    cb_delta.set_label("Delta (Q - C)")
    plt.show()

last_surface = row_to_surface(train_df.iloc[-1])
plt.figure(figsize=(11, 6))
im = plt.imshow(last_surface, aspect="auto", interpolation="nearest", cmap="viridis")
plt.title(f"Last Training Surface ({train_df.iloc[-1]['Date'].date()})")
plt.xlabel("Maturity index")
plt.ylabel("Tenor index")
plt.colorbar(im, label="Price")
plt.tight_layout()
plt.show()


In [ ]:
pred_surface = row_to_surface(pred_df.iloc[0])

# This compares forecast vs last observed train surface (distribution shift view), not forecast error.
plot_reference_vs_prediction(
    last_surface,
    pred_surface,
    ref_title="Last Train Surface",
    pred_title="First Forecast (submission)",
    diff_title_prefix="Absolute Shift (not forecast error)",
)

# True error visualization on validation (no leakage), first validation sample only.
true_val_surface = array_to_surface(y_va_surface_eval[0])
pred_val_surface = array_to_surface(pred_va_surface_eval[0])
plot_reference_vs_prediction(
    true_val_surface,
    pred_val_surface,
    ref_title="Validation Ground Truth (t+1)",
    pred_title="Classical Prediction (t+1)",
    diff_title_prefix="Absolute Error (true)",
)


## Quantum vs Classical (MerLin/Perceval)
This section follows the Quandela tutorial style and builds a hybrid quantum readout to compare with the classical ridge baseline.


In [ ]:
HAS_QUANTUM = False
try:
    import torch
    from sklearn.decomposition import PCA as PCA_windows
    from merlin import QuantumLayer
    HAS_QUANTUM = True
    print("Quantum stack available.")
except Exception as e:
    print("Quantum stack not available, skipping quantum section:", repr(e))

if HAS_QUANTUM:
    split_idx = int(0.8 * len(X1))
    X_tr, X_va = X1[:split_idx], X1[split_idx:]
    y_tr, y_va = y1[:split_idx], y1[split_idx:]

    classical_cmp = Ridge(alpha=best_alpha)
    classical_cmp.fit(X_tr, y_tr)
    pred_va_classical = classical_cmp.predict(X_va)

    naive_va = X_va[:, -k:]
    mean_va = np.repeat(y_tr.mean(axis=0, keepdims=True), len(y_va), axis=0)

    y_va_surface = factors_to_surface(y_va)
    pred_va_surface_classical = factors_to_surface(pred_va_classical)
    naive_va_surface = factors_to_surface(naive_va)
    mean_va_surface = factors_to_surface(mean_va)

    d_q = min(16, X_tr.shape[1], 20)
    pca_q = PCA_windows(n_components=d_q, random_state=0)
    X_tr_q = pca_q.fit_transform(X_tr)
    X_va_q = pca_q.transform(X_va)

    qlayer = QuantumLayer.simple(input_size=X_tr_q.shape[1], dtype=torch.float32)

    with torch.no_grad():
        phi_tr = qlayer(torch.tensor(X_tr_q, dtype=torch.float32)).detach().cpu().numpy()
        phi_va = qlayer(torch.tensor(X_va_q, dtype=torch.float32)).detach().cpu().numpy()

    quantum_readout = Ridge(alpha=1.0)
    quantum_readout.fit(phi_tr, y_tr)
    pred_va_quantum = quantum_readout.predict(phi_va)

    pred_va_surface_quantum = factors_to_surface(pred_va_quantum)

    cmp_df = pd.DataFrame({
        "Model": [
            "Classical Ridge",
            "Naive persistence",
            "Mean predictor",
            "Quantum Hybrid (QLayer + Ridge)",
        ],
        "Factor_MSE": [
            mean_squared_error(y_va, pred_va_classical),
            mean_squared_error(y_va, naive_va),
            mean_squared_error(y_va, mean_va),
            mean_squared_error(y_va, pred_va_quantum),
        ],
        "Factor_MAE": [
            mean_absolute_error(y_va, pred_va_classical),
            mean_absolute_error(y_va, naive_va),
            mean_absolute_error(y_va, mean_va),
            mean_absolute_error(y_va, pred_va_quantum),
        ],
        "Surface_MSE": [
            mean_squared_error(y_va_surface, pred_va_surface_classical),
            mean_squared_error(y_va_surface, naive_va_surface),
            mean_squared_error(y_va_surface, mean_va_surface),
            mean_squared_error(y_va_surface, pred_va_surface_quantum),
        ],
        "Surface_MAE": [
            mean_absolute_error(y_va_surface, pred_va_surface_classical),
            mean_absolute_error(y_va_surface, naive_va_surface),
            mean_absolute_error(y_va_surface, mean_va_surface),
            mean_absolute_error(y_va_surface, pred_va_surface_quantum),
        ],
    })

    classical_factor_mse = float(cmp_df.loc[cmp_df['Model'] == 'Classical Ridge', 'Factor_MSE'].iloc[0])
    cmp_df['Factor_MSE_vs_Classical'] = cmp_df['Factor_MSE'] / classical_factor_mse

    try:
        display(cmp_df.round(8))
    except NameError:
        print(cmp_df.round(8).to_string(index=False))

    print("Quantum feature shape:", phi_tr.shape)
    print("Compression dim d_q:", d_q)


In [ ]:
if HAS_QUANTUM:
    def predict_factors_iterative_quantum(F_hist, steps, W, pca_windows, q_layer, readout):
        F_ext = F_hist.copy()
        preds = []
        for _ in range(steps):
            x = F_ext[-W:].reshape(1, -1)
            x_q = pca_windows.transform(x)
            with torch.no_grad():
                phi = q_layer(torch.tensor(x_q, dtype=torch.float32)).detach().cpu().numpy()
            f_next = readout.predict(phi)
            F_ext = np.vstack([F_ext, f_next])
            preds.append(f_next.reshape(-1))
        return np.vstack(preds)

    F_future_q = predict_factors_iterative_quantum(
        F_hist=F,
        steps=len(template_df),
        W=W,
        pca_windows=pca_q,
        q_layer=qlayer,
        readout=quantum_readout,
    )

    Y_future_q = factors_to_surface(F_future_q)
    pred_df_q = pd.DataFrame(Y_future_q, columns=surface_cols)[tmpl_surface_cols]

    submission_q = template_df.copy()
    submission_q.loc[:, tmpl_surface_cols] = pred_df_q.values
    submission_q_path = OUTPUT_DIR / "submission_quantum_hybrid_notebook.xlsx"
    submission_q.to_excel(submission_q_path, index=False)
    print("quantum submission saved to:", submission_q_path)


In [ ]:
if HAS_QUANTUM:
    q_surface = row_to_surface(pred_df_q.iloc[0])
    c_surface = row_to_surface(pred_df.iloc[0])
    plot_classical_vs_quantum(c_surface, q_surface)


## Executive Summary
Automatic performance checks against baselines.


In [ ]:
def _winner_row(df, metric_col):
    idx = df[metric_col].astype(float).idxmin()
    row = df.loc[idx]
    return row['Model'], float(row[metric_col])

def _status_icon(ok: bool) -> str:
    return 'OK' if ok else 'ALERT'

print('=== Executive Summary ===')

if 'metrics_df' in globals():
    print('\nValidation (no leakage)')
    metrics_local = metrics_df.copy()
    metrics_local = metrics_local.sort_values('Surface_MAE').reset_index(drop=True)
    try:
        display(metrics_local.round(8))
    except NameError:
        print(metrics_local.round(8).to_string(index=False))

    for metric in ['Factor_MSE', 'Factor_MAE', 'Surface_MSE', 'Surface_MAE']:
        model, value = _winner_row(metrics_local, metric)
        print(f'Best {metric}: {model} ({value:.8f})')

    naive_row = metrics_local.loc[metrics_local['Model'] == 'Naive persistence'].iloc[0]
    classical_row = metrics_local.loc[metrics_local['Model'] == 'Classical Ridge'].iloc[0]
    for metric in ['Factor_MSE', 'Factor_MAE', 'Surface_MSE', 'Surface_MAE']:
        classical_ok = float(classical_row[metric]) < float(naive_row[metric])
        print(f"{_status_icon(classical_ok)} | Classical beats naive on {metric}: {classical_ok}")

if 'recursive_df' in globals():
    print('\nRecursive walk-forward validation')
    rec_local = recursive_df.copy()
    rec_local = rec_local.sort_values('Surface_MAE').reset_index(drop=True)
    try:
        display(rec_local.round(8))
    except NameError:
        print(rec_local.round(8).to_string(index=False))

    for metric in ['Factor_MSE', 'Factor_MAE', 'Surface_MSE', 'Surface_MAE']:
        model, value = _winner_row(rec_local, metric)
        print(f'Best recursive {metric}: {model} ({value:.8f})')

if 'cmp_df' in globals():
    print('\nQuantum comparison (if available)')
    cmp_local = cmp_df.copy().sort_values('Surface_MAE').reset_index(drop=True)
    try:
        display(cmp_local.round(8))
    except NameError:
        print(cmp_local.round(8).to_string(index=False))

    if 'Naive persistence' in set(cmp_local['Model']) and 'Quantum Hybrid (QLayer + Ridge)' in set(cmp_local['Model']):
        naive_q = cmp_local.loc[cmp_local['Model'] == 'Naive persistence'].iloc[0]
        quantum_q = cmp_local.loc[cmp_local['Model'] == 'Quantum Hybrid (QLayer + Ridge)'].iloc[0]
        for metric in ['Factor_MSE', 'Factor_MAE', 'Surface_MSE', 'Surface_MAE']:
            quantum_ok = float(quantum_q[metric]) < float(naive_q[metric])
            print(f"{_status_icon(quantum_ok)} | Quantum beats naive on {metric}: {quantum_ok}")
